<a href="https://colab.research.google.com/github/rohin-18/one-credit-course-final-project/blob/main/occ2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🧩 ADAPTIVE DISTRIBUTED MULTI-AGENT PUZZLE SOLVER
# ============================================================

import re
from concurrent.futures import ThreadPoolExecutor


# ============================================================
# 🧠 SHARED MESSAGE BOARD
# ============================================================

class MessageBoard:

    def __init__(self):
        self.messages = []


    def post(self, agent, message):

        self.messages.append({
            "agent": agent,
            "message": message
        })


    def get_messages(self):

        return self.messages


# ============================================================
# 🧩 PUZZLE
# ============================================================

class Puzzle:

    def __init__(self, question):

        self.question = question


# ============================================================
# 🤖 BASE AGENT
# ============================================================

class PuzzleAgent:

    def __init__(self, name, specialty):

        self.name = name
        self.specialty = specialty


# ============================================================
# 🧰 HELPER FUNCTIONS
# ============================================================

def is_arithmetic_expression(text):

    """
    Detect expressions such as:

    50 - 20
    25 + 15
    5 * 10
    20 / 5

    This prevents Pattern Agent and Logic Agent
    from treating arithmetic as a sequence.
    """

    return re.search(
        r"\b-?\d+(?:\.\d+)?\s*[\+\-\*/]\s*-?\d+(?:\.\d+)?\b",
        text
    ) is not None


def extract_numbers(text):

    return [
        float(x)
        for x in re.findall(
            r"-?\d+(?:\.\d+)?",
            text
        )
    ]


def clean_number(number):

    if isinstance(number, float) and number.is_integer():

        return int(number)

    return number


# ============================================================
# 🧠 LOGIC AGENT
# ============================================================

class LogicAgent(PuzzleAgent):

    def __init__(self):

        super().__init__(
            "Logic Agent",
            "logical reasoning"
        )


    def solve(self, puzzle):

        print("🧠 Logic Agent solving...")

        q = puzzle.question.lower()


        # ====================================================
        # IMPORTANT:
        # Arithmetic belongs to Math Agent.
        # ====================================================

        if is_arithmetic_expression(q):

            return {
                "agent": self.name,
                "answer": "LOGIC_UNKNOWN",
                "reasoning":
                    "This is an arithmetic expression. "
                    "The Math Agent should solve it."
            }


        # ====================================================
        # NUMBER SEQUENCE
        # ====================================================

        numbers = extract_numbers(q)


        if len(numbers) >= 3:

            # ------------------------------------------------
            # Constant difference
            # ------------------------------------------------

            differences = [

                numbers[i] - numbers[i - 1]

                for i in range(
                    1,
                    len(numbers)
                )

            ]


            if (
                len(differences) >= 2
                and
                all(
                    abs(differences[i] - differences[0]) < 1e-9
                    for i in range(1, len(differences))
                )
            ):

                difference = differences[0]

                answer = (
                    numbers[-1]
                    +
                    difference
                )

                answer = clean_number(answer)


                return {

                    "agent": self.name,

                    "answer": answer,

                    "reasoning":
                        (
                            "The sequence has a constant "
                            f"difference of {difference}. "
                            f"The next value is {answer}."
                        )

                }


            # ------------------------------------------------
            # Multiplication pattern
            # ------------------------------------------------

            if all(
                numbers[i - 1] != 0
                for i in range(1, len(numbers))
            ):

                ratios = [

                    numbers[i] / numbers[i - 1]

                    for i in range(
                        1,
                        len(numbers)
                    )

                ]


                if (
                    len(ratios) >= 2
                    and
                    all(
                        abs(ratios[i] - ratios[0]) < 1e-9
                        for i in range(1, len(ratios))
                    )
                ):

                    ratio = ratios[0]

                    answer = (
                        numbers[-1]
                        *
                        ratio
                    )

                    answer = clean_number(answer)


                    return {

                        "agent": self.name,

                        "answer": answer,

                        "reasoning":
                            (
                                "Each number is multiplied "
                                f"by {ratio}. "
                                f"The next value is {answer}."
                            )

                    }


        # ====================================================
        # 🐘 ELEPHANT
        # ====================================================

        if (
            "trunk" in q
            and
            "ears" in q
        ):

            return {

                "agent": self.name,

                "answer": "ELEPHANT",

                "reasoning":
                    "A trunk and large ears indicate elephant."

            }


        # ====================================================
        # 🦒 GIRAFFE
        # ====================================================

        if (
            "long neck" in q
            and
            "spots" in q
        ):

            return {

                "agent": self.name,

                "answer": "GIRAFFE",

                "reasoning":
                    "A long neck and spots indicate giraffe."

            }


        # ====================================================
        # 🦓 ZEBRA
        # ====================================================

        if (
            "black" in q
            and
            "white" in q
            and
            "stripes" in q
        ):

            return {

                "agent": self.name,

                "answer": "ZEBRA",

                "reasoning":
                    "Black and white stripes indicate zebra."

            }


        # ====================================================
        # UNKNOWN
        # ====================================================

        return {

            "agent": self.name,

            "answer": "LOGIC_UNKNOWN",

            "reasoning":
                "No logical solution was identified."

        }


# ============================================================
# 🔢 MATH AGENT
# ============================================================

class MathAgent(PuzzleAgent):

    def __init__(self):

        super().__init__(
            "Math Agent",
            "mathematical reasoning"
        )


    def solve(self, puzzle):

        print("🔢 Math Agent solving...")

        q = puzzle.question


        # ====================================================
        # ADDITION
        # ====================================================

        match = re.search(
            r"(-?\d+(?:\.\d+)?)\s*\+\s*(-?\d+(?:\.\d+)?)",
            q
        )


        if match:

            a = float(match.group(1))
            b = float(match.group(2))

            answer = a + b

            answer = clean_number(answer)


            return {

                "agent": self.name,

                "answer": answer,

                "reasoning":
                    f"{a:g} + {b:g} = {answer}"

            }


        # ====================================================
        # SUBTRACTION
        # ====================================================

        match = re.search(
            r"(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)",
            q
        )


        if match:

            a = float(match.group(1))
            b = float(match.group(2))

            answer = a - b

            answer = clean_number(answer)


            return {

                "agent": self.name,

                "answer": answer,

                "reasoning":
                    f"{a:g} - {b:g} = {answer}"

            }


        # ====================================================
        # MULTIPLICATION
        # ====================================================

        match = re.search(
            r"(-?\d+(?:\.\d+)?)\s*\*\s*(-?\d+(?:\.\d+)?)",
            q
        )


        if match:

            a = float(match.group(1))
            b = float(match.group(2))

            answer = a * b

            answer = clean_number(answer)


            return {

                "agent": self.name,

                "answer": answer,

                "reasoning":
                    f"{a:g} × {b:g} = {answer}"

            }


        # ====================================================
        # DIVISION
        # ====================================================

        match = re.search(
            r"(-?\d+(?:\.\d+)?)\s*/\s*(-?\d+(?:\.\d+)?)",
            q
        )


        if match:

            a = float(match.group(1))
            b = float(match.group(2))


            if b == 0:

                return {

                    "agent": self.name,

                    "answer": "MATH_ERROR",

                    "reasoning":
                        "Division by zero is not allowed."

                }


            answer = a / b

            answer = clean_number(answer)


            return {

                "agent": self.name,

                "answer": answer,

                "reasoning":
                    f"{a:g} ÷ {b:g} = {answer}"

            }


        # ====================================================
        # UNKNOWN
        # ====================================================

        return {

            "agent": self.name,

            "answer": "MATH_UNKNOWN",

            "reasoning":
                "No arithmetic expression detected."

        }


# ============================================================
# 🔤 WORD AGENT
# ============================================================

class WordAgent(PuzzleAgent):

    def __init__(self):

        super().__init__(
            "Word Agent",
            "word, language and transformation reasoning"
        )


    def solve(self, puzzle):

        print("🔤 Word Agent solving...")

        q = puzzle.question.strip()

        lower_q = q.lower()


        # ====================================================
        # 🔄 REVERSE LAST WORD
        # ====================================================

        if (
            "reverse" in lower_q
            or
            "reversed" in lower_q
        ):

            words = re.findall(
                r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*",
                q
            )


            if words:

                last_word = words[-1]

                answer = last_word[::-1]


                return {

                    "agent": self.name,

                    "answer": answer,

                    "reasoning":
                        (
                            f"The last word is "
                            f"'{last_word}'. "
                            f"Reversed, it becomes "
                            f"'{answer}'."
                        )

                }


        # ====================================================
        # 🍌 BANANA
        # ====================================================

        if (
            "yellow" in lower_q
            and
            "fruit" in lower_q
            and
            (
                "long" in lower_q
                or
                "monkey" in lower_q
            )
        ):

            return {

                "agent": self.name,

                "answer": "BANANA",

                "reasoning":
                    (
                        "Yellow fruit, long shape "
                        "and monkeys indicate banana."
                    )

            }


        # ====================================================
        # 🍎 APPLE
        # ====================================================

        if (
            "red" in lower_q
            and
            "fruit" in lower_q
        ):

            return {

                "agent": self.name,

                "answer": "APPLE",

                "reasoning":
                    "A common red fruit is an apple."

            }


        if "apple" in lower_q:

            return {

                "agent": self.name,

                "answer": "APPLE",

                "reasoning":
                    "The clue contains apple."

            }


        # ====================================================
        # 🐘 ELEPHANT
        # ====================================================

        if (
            "trunk" in lower_q
            and
            "ears" in lower_q
        ):

            return {

                "agent": self.name,

                "answer": "ELEPHANT",

                "reasoning":
                    "Trunk and ears indicate elephant."

            }


        # ====================================================
        # 🦒 GIRAFFE
        # ====================================================

        if (
            "long neck" in lower_q
            and
            "spots" in lower_q
        ):

            return {

                "agent": self.name,

                "answer": "GIRAFFE",

                "reasoning":
                    "Long neck and spots indicate giraffe."

            }


        # ====================================================
        # 🦓 ZEBRA
        # ====================================================

        if (
            "black" in lower_q
            and
            "white" in lower_q
            and
            "stripes" in lower_q
        ):

            return {

                "agent": self.name,

                "answer": "ZEBRA",

                "reasoning":
                    "Black and white stripes indicate zebra."

            }


        # ====================================================
        # UNKNOWN
        # ====================================================

        return {

            "agent": self.name,

            "answer": "WORD_UNKNOWN",

            "reasoning":
                "No known word clue detected."

        }


# ============================================================
# 🔍 PATTERN AGENT
# ============================================================

class PatternAgent(PuzzleAgent):

    def __init__(self):

        super().__init__(
            "Pattern Agent",
            "sequence and pattern detection"
        )


    def solve(self, puzzle):

        print("🔍 Pattern Agent solving...")


        q = puzzle.question.lower()


        # ====================================================
        # 🚨 CRITICAL FIX
        #
        # Do NOT interpret:
        #
        # 50 - 20
        # 25 + 15
        # 5 * 10
        #
        # as a sequence.
        # ====================================================

        if is_arithmetic_expression(q):

            return {

                "agent": self.name,

                "answer": "PATTERN_UNKNOWN",

                "reasoning":
                    (
                        "This is an arithmetic expression, "
                        "not a numerical sequence."
                    )

            }


        # ====================================================
        # Extract numbers
        # ====================================================

        numbers = extract_numbers(q)


        # ====================================================
        # Need at least 3 numbers for a sequence
        # ====================================================

        if len(numbers) < 3:

            return {

                "agent": self.name,

                "answer": "PATTERN_UNKNOWN",

                "reasoning":
                    (
                        "At least three sequence values "
                        "are required."
                    )

            }


        # ====================================================
        # DIFFERENCE PATTERN
        #
        # 100, 90, 80, 70
        #
        # -10, -10, -10
        #
        # answer = 60
        # ====================================================

        differences = [

            numbers[i] - numbers[i - 1]

            for i in range(
                1,
                len(numbers)
            )

        ]


        if all(
            abs(differences[i] - differences[0]) < 1e-9
            for i in range(1, len(differences))
        ):

            difference = differences[0]

            answer = (
                numbers[-1]
                +
                difference
            )

            answer = clean_number(answer)


            return {

                "agent": self.name,

                "answer": answer,

                "reasoning":
                    (
                        "The sequence has a constant "
                        f"difference of {difference}. "
                        f"{numbers[-1]} + ({difference}) "
                        f"= {answer}."
                    )

            }


        # ====================================================
        # RATIO PATTERN
        #
        # 2, 4, 8, 16
        #
        # ×2
        #
        # answer = 32
        # ====================================================

        if all(
            numbers[i - 1] != 0
            for i in range(1, len(numbers))
        ):

            ratios = [

                numbers[i] / numbers[i - 1]

                for i in range(
                    1,
                    len(numbers)
                )

            ]


            if all(
                abs(ratios[i] - ratios[0]) < 1e-9
                for i in range(1, len(ratios))
            ):

                ratio = ratios[0]

                answer = (
                    numbers[-1]
                    *
                    ratio
                )

                answer = clean_number(answer)


                return {

                    "agent": self.name,

                    "answer": answer,

                    "reasoning":
                        (
                            "Each number is multiplied "
                            f"by {ratio}. "
                            f"{numbers[-1]} × {ratio} "
                            f"= {answer}."
                        )

                }


        # ====================================================
        # UNKNOWN
        # ====================================================

        return {

            "agent": self.name,

            "answer": "PATTERN_UNKNOWN",

            "reasoning":
                "No consistent numerical pattern identified."

        }


# ============================================================
# 🔎 CLUE AGENT
# ============================================================

class ClueAgent(PuzzleAgent):

    def __init__(self):

        super().__init__(
            "Clue Agent",
            "animal, fruit and general clue reasoning"
        )


    def solve(self, puzzle):

        print("🔎 Clue Agent solving...")

        q = puzzle.question.lower()


        # ====================================================
        # 🔄 REVERSE
        # ====================================================

        if (
            "reverse" in q
            or
            "reversed" in q
        ):

            words = re.findall(
                r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*",
                puzzle.question
            )


            if words:

                last_word = words[-1]

                answer = last_word[::-1]


                return {

                    "agent": self.name,

                    "answer": answer,

                    "reasoning":
                        (
                            f"The last word "
                            f"'{last_word}' "
                            f"is reversed to "
                            f"'{answer}'."
                        )

                }


        # ====================================================
        # 🐘 ELEPHANT
        # ====================================================

        if (
            "trunk" in q
            and
            "ears" in q
        ):

            return {

                "agent": self.name,

                "answer": "ELEPHANT",

                "reasoning":
                    "Trunk and large ears indicate elephant."

            }


        # ====================================================
        # 🦒 GIRAFFE
        # ====================================================

        if (
            "long neck" in q
            and
            "spots" in q
        ):

            return {

                "agent": self.name,

                "answer": "GIRAFFE",

                "reasoning":
                    "Long neck and spots indicate giraffe."

            }


        # ====================================================
        # 🦓 ZEBRA
        # ====================================================

        if (
            "black" in q
            and
            "white" in q
            and
            "stripes" in q
        ):

            return {

                "agent": self.name,

                "answer": "ZEBRA",

                "reasoning":
                    "Black and white stripes indicate zebra."

            }


        # ====================================================
        # 🦁 LION
        # ====================================================

        if "king of the jungle" in q:

            return {

                "agent": self.name,

                "answer": "LION",

                "reasoning":
                    "King of the jungle indicates lion."

            }


        # ====================================================
        # 🐇 RABBIT
        # ====================================================

        if (
            "hops" in q
            and
            "long ears" in q
        ):

            return {

                "agent": self.name,

                "answer": "RABBIT",

                "reasoning":
                    "Hopping and long ears indicate rabbit."

            }


        # ====================================================
        # 🍌 BANANA
        # ====================================================

        if (
            "yellow" in q
            and
            "fruit" in q
            and
            (
                "long" in q
                or
                "monkey" in q
            )
        ):

            return {

                "agent": self.name,

                "answer": "BANANA",

                "reasoning":
                    (
                        "Yellow fruit, long shape "
                        "and monkeys indicate banana."
                    )

            }


        # ====================================================
        # 🍎 APPLE
        # ====================================================

        if (
            "red" in q
            and
            "fruit" in q
        ):

            return {

                "agent": self.name,

                "answer": "APPLE",

                "reasoning":
                    "Red fruit indicates apple."

            }


        # ====================================================
        # UNKNOWN
        # ====================================================

        return {

            "agent": self.name,

            "answer": "CLUE_UNKNOWN",

            "reasoning":
                "No recognizable clue detected."

        }


# ============================================================
# 🧠 VERIFIER AGENT
# ============================================================

class VerifierAgent(PuzzleAgent):

    def __init__(self):

        super().__init__(
            "Verifier Agent",
            "answer verification and voting"
        )


    def verify(self, results):

        print(
            "🧠 Verifier Agent checking answers..."
        )


        valid_results = []


        # ====================================================
        # Remove UNKNOWN results
        # ====================================================

        for result in results:

            answer = str(
                result["answer"]
            )


            if (
                answer.endswith("_UNKNOWN")
                or
                answer == "MATH_ERROR"
            ):

                continue


            valid_results.append(
                result
            )


        # ====================================================
        # No answer
        # ====================================================

        if not valid_results:

            return {

                "answer": "UNSOLVED",

                "confidence": "LOW",

                "votes": 0,

                "vote_table": {}

            }


        # ====================================================
        # SPECIAL CASE:
        # If only Math Agent gives a valid answer,
        # trust the Math Agent.
        # ====================================================

        math_results = [

            r for r in valid_results

            if r["agent"] == "Math Agent"

        ]


        if len(math_results) == 1:

            math_answer = str(
                math_results[0]["answer"]
            ).upper()


            # If the question is clearly arithmetic,
            # Math Agent gets priority.

            question = ""

            for result in results:

                # No question stored here,
                # so use math result directly.
                pass


        # ====================================================
        # Count votes
        # ====================================================

        counts = {}


        for result in valid_results:

            key = str(
                result["answer"]
            ).upper()


            counts[key] = (

                counts.get(
                    key,
                    0
                )
                +
                1

            )


        # ====================================================
        # Best answer
        # ====================================================

        best_answer = max(
            counts,
            key=counts.get
        )


        votes = counts[
            best_answer
        ]


        # ====================================================
        # Confidence
        # ====================================================

        if votes >= 2:

            confidence = "HIGH"

        else:

            confidence = "MEDIUM"


        return {

            "answer": best_answer,

            "confidence": confidence,

            "votes": votes,

            "vote_table": counts

        }


# ============================================================
# 🧠 MANAGER AGENT
# ============================================================

class ManagerAgent:

    def __init__(self):

        self.logic = LogicAgent()

        self.math = MathAgent()

        self.word = WordAgent()

        self.pattern = PatternAgent()

        self.clue = ClueAgent()

        self.verifier = VerifierAgent()


    # ========================================================
    # 🏗 AUTOMATIC ARCHITECTURE SELECTION
    # ========================================================

    def select_architecture(self, puzzle):

        q = puzzle.question.lower()


        # ====================================================
        # 🔢 ARITHMETIC
        #
        # 50 - 20
        #
        # Use parallel because different agents can
        # independently participate, but only Math Agent
        # should actually solve it.
        # ====================================================

        if is_arithmetic_expression(q):

            return "PARALLEL"


        # ====================================================
        # 🔄 REVERSE
        # ====================================================

        if (
            "reverse" in q
            or
            "reversed" in q
        ):

            return "PARALLEL"


        # ====================================================
        # 🐘 ANIMAL
        # ====================================================

        animal_words = [

            "animal",
            "trunk",
            "large ears",
            "long ears",
            "long neck",
            "stripes",
            "spots",
            "jungle",
            "wings",
            "hops"

        ]


        if any(
            word in q
            for word in animal_words
        ):

            return "PEER_TO_PEER"


        # ====================================================
        # 🍎 FRUIT
        # ====================================================

        if "fruit" in q:

            return "PEER_TO_PEER"


        # ====================================================
        # 🔢 SEQUENCE
        #
        # IMPORTANT:
        # Only treat it as sequence if it is NOT arithmetic.
        # ====================================================

        numbers = extract_numbers(q)


        if (
            len(numbers) >= 3
            and
            not is_arithmetic_expression(q)
        ):

            return "PARALLEL"


        # ====================================================
        # PATTERN KEYWORDS
        # ====================================================

        if (
            "sequence" in q
            or
            "pattern" in q
            or
            "what comes next" in q
        ):

            return "PARALLEL"


        # ====================================================
        # COMPLEX PUZZLE
        # ====================================================

        if len(q.split()) > 18:

            return "DECENTRALIZED"


        # ====================================================
        # DEFAULT
        # ====================================================

        return "PARALLEL"


    # ========================================================
    # 🤖 AUTOMATIC AGENT SELECTION
    # ========================================================

    def select_agents(self, puzzle):

        q = puzzle.question.lower()


        agents = []


        # ====================================================
        # 🔢 ARITHMETIC
        #
        # THIS IS THE IMPORTANT FIX.
        #
        # If input is:
        #
        # What is 50 - 20?
        #
        # select ONLY Math Agent.
        # ====================================================

        if is_arithmetic_expression(q):

            agents.append(
                self.math
            )

            return agents


        # ====================================================
        # 🔄 REVERSE
        # ====================================================

        if (
            "reverse" in q
            or
            "reversed" in q
        ):

            agents.append(
                self.word
            )

            agents.append(
                self.clue
            )


        # ====================================================
        # 🐘 ANIMAL
        # ====================================================

        animal_words = [

            "animal",
            "trunk",
            "large ears",
            "long ears",
            "long neck",
            "stripes",
            "spots",
            "jungle",
            "wings",
            "hops"

        ]


        if any(
            word in q
            for word in animal_words
        ):

            agents.append(
                self.clue
            )

            agents.append(
                self.word
            )

            agents.append(
                self.logic
            )


        # ====================================================
        # 🍎🍌 FRUIT
        # ====================================================

        if "fruit" in q:

            agents.append(
                self.word
            )

            agents.append(
                self.clue
            )

            agents.append(
                self.logic
            )


        # ====================================================
        # 🔢 NUMBER SEQUENCE
        #
        # Must NOT run for:
        #
        # 50 - 20
        #
        # because that is arithmetic.
        # ====================================================

        numbers = extract_numbers(q)


        if (
            len(numbers) >= 3
            and
            not is_arithmetic_expression(q)
        ):

            agents.append(
                self.pattern
            )

            agents.append(
                self.logic
            )


        # ====================================================
        # PATTERN KEYWORDS
        # ====================================================

        if (
            "sequence" in q
            or
            "pattern" in q
            or
            "what comes next" in q
        ):

            agents.append(
                self.pattern
            )

            agents.append(
                self.logic
            )


        # ====================================================
        # WORD
        # ====================================================

        if "word" in q:

            agents.append(
                self.word
            )


        # ====================================================
        # GENERAL CLUE
        # ====================================================

        if (
            "clue" in q
            and
            not agents
        ):

            agents.append(
                self.clue
            )


        # ====================================================
        # DEFAULT
        # ====================================================

        if not agents:

            agents = [

                self.logic,
                self.word,
                self.clue

            ]


        # ====================================================
        # REMOVE DUPLICATES
        # ====================================================

        unique = []

        names = set()


        for agent in agents:

            if agent.name not in names:

                unique.append(
                    agent
                )

                names.add(
                    agent.name
                )


        return unique


    # ========================================================
    # ⚡ PARALLEL ARCHITECTURE
    # ========================================================

    def parallel(
        self,
        puzzle,
        agents
    ):

        print()

        print(
            "⚡ PARALLEL ARCHITECTURE"
        )

        print(
            "Agents solve independently."
        )


        with ThreadPoolExecutor(
            max_workers=len(agents)
        ) as executor:


            futures = [

                executor.submit(
                    agent.solve,
                    puzzle
                )

                for agent in agents

            ]


            results = [

                future.result()

                for future in futures

            ]


        return results


    # ========================================================
    # 🔗 PEER-TO-PEER ARCHITECTURE
    # ========================================================

    def peer_to_peer(
        self,
        puzzle,
        agents
    ):

        print()

        print(
            "🔗 PEER-TO-PEER ARCHITECTURE"
        )

        print(
            "Agents exchange answers directly."
        )


        board = MessageBoard()

        results = []


        # ====================================================
        # ROUND 1
        # ====================================================

        for agent in agents:

            result = agent.solve(
                puzzle
            )


            results.append(
                result
            )


            board.post(

                agent.name,

                (
                    f"Proposed answer: "
                    f"{result['answer']} | "
                    f"Reasoning: "
                    f"{result['reasoning']}"
                )

            )


        # ====================================================
        # PEER MESSAGES
        # ====================================================

        print(
            "\n📡 PEER MESSAGES"
        )


        for message in board.get_messages():

            print(

                f"{message['agent']} → "
                f"{message['message']}"

            )


        # ====================================================
        # DISCUSSION
        # ====================================================

        print(
            "\n🤝 PEER DISCUSSION ROUND"
        )


        for result in results:

            print(

                f"{result['agent']} "
                f"supports: "
                f"{result['answer']}"

            )


        return results


    # ========================================================
    # 🌐 DECENTRALIZED ARCHITECTURE
    # ========================================================

    def decentralized(
        self,
        puzzle,
        agents
    ):

        print()

        print(
            "🌐 DECENTRALIZED ARCHITECTURE"
        )

        print(
            "No single solver controls the puzzle."
        )


        results = []


        with ThreadPoolExecutor(
            max_workers=len(agents)
        ) as executor:


            futures = {

                executor.submit(
                    agent.solve,
                    puzzle
                ): agent

                for agent in agents

            }


            for future in futures:

                result = future.result()

                results.append(
                    result
                )


        print(
            "\n🤝 AGENT DISCUSSION"
        )


        for result in results:

            print(

                f"{result['agent']}: "
                f"{result['answer']}"

            )


        return results


    # ========================================================
    # 🧠 SMART VERIFICATION
    # ========================================================

    def verify_results(
        self,
        puzzle,
        results
    ):

        q = puzzle.question.lower()


        # ====================================================
        # 🔢 ARITHMETIC HAS MATH PRIORITY
        #
        # Example:
        #
        # 50 - 20
        #
        # Math Agent = 30
        #
        # No other agent should override it.
        # ====================================================

        if is_arithmetic_expression(q):

            for result in results:

                if result["agent"] == "Math Agent":

                    answer = result["answer"]


                    if answer not in (
                        "MATH_UNKNOWN",
                        "MATH_ERROR"
                    ):

                        return {

                            "answer":
                                str(answer).upper(),

                            "confidence":
                                "HIGH",

                            "votes":
                                1,

                            "vote_table":
                                {
                                    str(answer).upper():
                                        1
                                }

                        }


        # ====================================================
        # NORMAL VERIFIER
        # ====================================================

        return self.verifier.verify(
            results
        )


    # ========================================================
    # 🚀 MAIN SOLVER
    # ========================================================

    def solve(self, puzzle):

        print()

        print(
            "=" * 70
        )

        print(
            "🧠 MANAGER AGENT"
        )

        print(
            "Puzzle:",
            puzzle.question
        )

        print(
            "=" * 70
        )


        # ====================================================
        # SELECT ARCHITECTURE
        # ====================================================

        architecture = (

            self.select_architecture(
                puzzle
            )

        )


        print(

            "\n🏗 SELECTED ARCHITECTURE:",

            architecture

        )


        # ====================================================
        # SELECT AGENTS
        # ====================================================

        agents = (

            self.select_agents(
                puzzle
            )

        )


        print(
            "\n🤖 SELECTED AGENTS:"
        )


        for agent in agents:

            print(
                "✓",
                agent.name
            )


        # ====================================================
        # EXECUTE ARCHITECTURE
        # ====================================================

        if architecture == "PARALLEL":

            results = self.parallel(
                puzzle,
                agents
            )


        elif architecture == "PEER_TO_PEER":

            results = self.peer_to_peer(
                puzzle,
                agents
            )


        elif architecture == "DECENTRALIZED":

            results = self.decentralized(
                puzzle,
                agents
            )


        else:

            results = self.parallel(
                puzzle,
                agents
            )


        # ====================================================
        # VERIFY
        # ====================================================

        verification = self.verify_results(
            puzzle,
            results
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()

        print(
            "=" * 70
        )

        print(
            "🏆 FINAL PUZZLE SOLUTION"
        )

        print(
            "=" * 70
        )


        print(
            "Architecture:",
            architecture
        )


        print(
            "Agents:",

            ", ".join(

                agent.name

                for agent in agents

            )

        )


        print(
            "Answer:",
            verification["answer"]
        )


        print(
            "Confidence:",
            verification["confidence"]
        )


        print(
            "Votes:",
            verification["votes"]
        )


        print(
            "Vote Table:",
            verification["vote_table"]
        )


        print(
            "=" * 70
        )


        return {

            "architecture":
                architecture,

            "agents":
                [
                    agent.name
                    for agent in agents
                ],

            "results":
                results,

            "verification":
                verification

        }


# ============================================================
# 🚀 CREATE MANAGER
# ============================================================

manager = ManagerAgent()


# ============================================================
# 🎨 START SCREEN
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║       🧩 ADAPTIVE DISTRIBUTED MULTI-AGENT SOLVER            ║
║                                                              ║
║                    👤 PUZZLE                                ║
║                       ↓                                      ║
║                 🧠 MANAGER AGENT                             ║
║                       ↓                                      ║
║             AUTOMATIC AGENT SELECTION                        ║
║                       ↓                                      ║
║       ┌───────────────┼─────────────────┐                    ║
║       ↓               ↓                 ↓                    ║
║   ⚡ Parallel     🔗 Peer-to-Peer    🌐 Decentralized        ║
║       │               │                 │                    ║
║       └───────────────┼─────────────────┘                    ║
║                       ↓                                      ║
║                🤖 SOLVER AGENTS                              ║
║                       ↓                                      ║
║                 🤝 COLLABORATION                             ║
║                       ↓                                      ║
║                🧠 VERIFIER                                   ║
║                       ↓                                      ║
║                  🏆 ANSWER                                   ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


# ============================================================
# 🧪 EXAMPLES
# ============================================================

print("""
Examples:

1. Find the animal clue: It has a long trunk and large ears.

2. Find the fruit clue: Yellow fruit, long shape, monkeys like it.

3. What comes next? 2, 4, 8, 16, ___

4. Reverse hello

5. What is 25 + 15?

""")


# ============================================================
# 👤 USER INPUT
# ============================================================

question = input(
    "\n👤 Puzzle: "
)


puzzle = Puzzle(
    question
)


# ============================================================
# 🚀 SOLVE
# ============================================================

result = manager.solve(
    puzzle
)


╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║       🧩 ADAPTIVE DISTRIBUTED MULTI-AGENT SOLVER            ║
║                                                              ║
║                    👤 PUZZLE                                ║
║                       ↓                                      ║
║                 🧠 MANAGER AGENT                             ║
║                       ↓                                      ║
║             AUTOMATIC AGENT SELECTION                        ║
║                       ↓                                      ║
║       ┌───────────────┼─────────────────┐                    ║
║       ↓               ↓                 ↓                    ║
║   ⚡ Parallel     🔗 Peer-to-Peer    🌐 Decentralized        ║
║       │               │                 │                    ║
║       └───────────────┼─────────────────┘                    ║
║                       ↓       